# Round 12: crossed-column target encoding, config diversity on the TE base - S6E10

Round 11 probed the Exp 31 winner's sensitivity: no variant (m=20, m=500, continuous-only TE)
cleared the decision heuristic vs the Exp 31 bar in either direction - the TE family remains
descriptively above baseline, m=100 stands, and Exp 31 stays the preferred candidate (highest
measured OOF is Exp 33a's 0.95994, but the guard promotes only on a working-signal win). Two
bounded tests on the Exp 31 scheme, one change each:

- **Exp 35: TE on crossed columns.** Value-level TE captures one column at a time. Airline
  satisfaction plausibly has interaction structure - Business travel x Business class is a
  different product from Personal travel x Eco. Two crossed columns (Class x Type of Travel,
  Customer Type x Class) target-encoded with the same leak-free construction; the raw crossed
  strings are dropped, only their encodings enter. Counts stay on the raw 21.
- **Exp 36: config diversity on the TE base.** Exp 32 (leaves {63, 127} x 3 seeds on the
  counts-only base) showed no detected improvement over Exp 29. Retest the same diversity on the
  stronger TE base: one change from Exp 31, 6-model average.

Construction, bar, and reading rules unchanged: split-first leak-free TE, the bar is Exp 31's
round-10 fold constants, paired per-fold deltas, 2x-spread band as a decision heuristic (never
proof), incumbent recomputed in-run. Refit fires only on a working-signal win over the Exp 31
bar. The 0.960 target is a mean-OOF target - single folds above it are not attainment.

In [ ]:
import inspect
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold, train_test_split

RANDOM_STATE = 42
N_SPLITS = 5
CATEGORICAL = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
CROSS_PAIRS = [('Class', 'Type of Travel'), ('Customer Type', 'Class')]
INCUMBENT_FOLDS_R3 = [0.95851, 0.95751, 0.95907, 0.95841, 0.95885]  # determinism reference
E22_FOLDS_R6 = [0.95886, 0.95756, 0.95953, 0.95875, 0.95922]        # stochastic averaging, round 6
E29_FOLDS_R9 = [0.95938, 0.95804, 0.96033, 0.95933, 0.95956]        # counts + stochastic, round 9
E31_FOLDS_R10 = [0.95998, 0.95873, 0.96056, 0.96000, 0.96000]       # counts + TE, the bar
STOCH = dict(bagging_fraction=0.9, bagging_freq=1, feature_fraction=0.9)
TE_SMOOTH = 100

In [ ]:
competition_dir = Path('.')
if not (competition_dir / 'data' / 'train.csv').is_file():
    if Path('playground-series-s6e10/data/train.csv').is_file():
        competition_dir = Path('playground-series-s6e10')
    elif Path('../data/train.csv').is_file():
        competition_dir = Path('..')
data_dir = competition_dir / 'data'

train = pd.read_csv(data_dir / 'train.csv')

def to_binary(series):
    if series.dtype == bool:
        return series.astype(int)
    return series.map({'True': 1, 'False': 0, True: 1, False: 0}).astype(int)

target = to_binary(train['satisfaction'])
features = train.drop(columns=['id', 'satisfaction'])
COUNT_COLS = list(features.columns)
TE_COLS_ALL = COUNT_COLS

def cross_name(pair):
    return ' x '.join(pair)

CROSS_COLS = [cross_name(p) for p in CROSS_PAIRS]

def add_crosses(df):
    out = df.copy()
    for pair in CROSS_PAIRS:
        out[cross_name(pair)] = out[pair[0]].astype(str) + ' x ' + out[pair[1]].astype(str)
    return out

def to_lgbm(df):
    out = df.copy()
    for col in CATEGORICAL:
        out[col] = out[col].astype('category')
    return out

def add_counts(train_part, other_parts=()):
    tables = {c: train_part[c].value_counts() for c in COUNT_COLS}
    def prep(df):
        out = df.copy()
        for c in COUNT_COLS:
            out[c + '_cnt'] = out[c].map(tables[c]).fillna(0).astype(np.int64)
        return out
    return prep(train_part), [prep(p) for p in other_parts]

def te_table(values, labels, m, gm):
    # smoothed value-level target mean: (sum + m*global) / (count + m)
    stats = labels.groupby(values).agg(['sum', 'count'])
    return (stats['sum'] + m * gm) / (stats['count'] + m)

def te_crossfit(X, y, cols, m=TE_SMOOTH, n_inner=5):
    # cross-fitted encodings: a row's own label never enters its encoding - not through the
    # value table and not through the smoothing mean (recomputed per inner fold from itr rows).
    # Inner folds are label-blind (plain shuffled KFold). (Round-10 audit-fixed construction.)
    te = pd.DataFrame(index=X.index)
    inner = KFold(n_splits=n_inner, shuffle=True, random_state=RANDOM_STATE)
    splits = list(inner.split(np.zeros(len(X))))
    for col in cols:
        enc = np.empty(len(X))
        for itr, iva in splits:
            gm = float(y.iloc[itr].mean())
            t = te_table(X[col].iloc[itr], y.iloc[itr], m, gm)
            enc[iva] = X[col].iloc[iva].map(t).fillna(gm).to_numpy()
        te[col + '_te'] = enc
    return te

def te_apply(X_ref, y_ref, X_new, cols, m=TE_SMOOTH):
    gm = float(y_ref.mean())
    te = pd.DataFrame(index=X_new.index)
    for col in cols:
        t = te_table(X_ref[col], y_ref, m, gm)
        te[col + '_te'] = X_new[col].map(t).fillna(gm).to_numpy()
    return te

def oof_lgbm(feat_frame, label, name, seed=RANDOM_STATE, count_features=False,
             target_encode=False, te_m=TE_SMOOTH, te_cols=None, te_drop=(), **params):
    oof = np.zeros(len(feat_frame))
    aucs, iters = [], []
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)  # same folds every time
    for fold, (tr_idx, va_idx) in enumerate(skf.split(feat_frame, label), 1):
        X_tr, y_tr = feat_frame.iloc[tr_idx], label.iloc[tr_idx]
        X_va, y_va = feat_frame.iloc[va_idx], label.iloc[va_idx]
        if count_features:
            X_tr, (X_va,) = add_counts(X_tr, [X_va])
        # early-stop split happens BEFORE any target encoding, so no early-stop label can
        # touch any early-stop encoding (round-10 audit fix)
        X_fit, X_es, y_fit, y_es = train_test_split(
            X_tr, y_tr, test_size=0.1, stratify=y_tr, random_state=RANDOM_STATE)
        if target_encode:
            cols = te_cols if te_cols is not None else TE_COLS_ALL
            te_fit = te_crossfit(X_fit, y_fit, cols, m=te_m)
            te_es = te_apply(X_fit, y_fit, X_es, cols, m=te_m)
            te_va = te_apply(X_tr, y_tr, X_va, cols, m=te_m)
            X_fit = pd.concat([X_fit.reset_index(drop=True), te_fit.reset_index(drop=True)], axis=1)
            X_es = pd.concat([X_es.reset_index(drop=True), te_es.reset_index(drop=True)], axis=1)
            X_va = pd.concat([X_va.reset_index(drop=True), te_va.reset_index(drop=True)], axis=1)
            if te_drop:  # raw crossed strings never enter the model - only their encodings
                X_fit = X_fit.drop(columns=list(te_drop))
                X_es = X_es.drop(columns=list(te_drop))
                X_va = X_va.drop(columns=list(te_drop))
        X_fit, X_es, X_va = to_lgbm(X_fit), to_lgbm(X_es), to_lgbm(X_va)
        # caller params merge over defaults (round-6 disclosed fix - never pass a key twice)
        cfg = dict(learning_rate=0.05, num_leaves=63, n_estimators=1000,
                   n_jobs=-1, random_state=seed, verbose=-1)
        cfg.update(params)
        model = lgb.LGBMClassifier(**cfg)
        # LightGBM 4.7.0 deprecated eval_set in favour of keyword-only eval_X/eval_y;
        # use whichever signature the installed version supports (no warning either way).
        if 'eval_X' in inspect.signature(lgb.LGBMClassifier.fit).parameters:
            model.fit(X_fit, y_fit, eval_X=X_es, eval_y=y_es, eval_metric='auc',
                      callbacks=[lgb.early_stopping(50, verbose=False)])
        else:
            model.fit(X_fit, y_fit, eval_set=[(X_es, y_es)], eval_metric='auc',
                      callbacks=[lgb.early_stopping(50, verbose=False)])
        oof[va_idx] = model.predict_proba(X_va)[:, 1]
        a = roc_auc_score(y_va, oof[va_idx])
        aucs.append(a); iters.append(model.best_iteration_)
        print(f'  {name} fold {fold}: AUC {a:.5f} (best_iteration={model.best_iteration_})')
    return oof, aucs, iters

def fold_aucs(oof, label):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    return [roc_auc_score(label.iloc[va], oof[va]) for _, va in skf.split(oof.reshape(-1, 1), label)]

print(f'Full train: {len(features):,} rows, {features.shape[1]} raw features; crossed columns: {CROSS_COLS}')

## Reference: incumbent recompute

Same-run incumbent with the determinism check, as every round.

In [ ]:
print('Reference: incumbent raw LightGBM (Exp 8 config), recomputed in-run')
oof_inc, aucs_inc, iters_inc = oof_lgbm(features, target, 'INC')
diffs = [abs(a - b) for a, b in zip(aucs_inc, INCUMBENT_FOLDS_R3)]
det_note = 'PASS - identical folds' if max(diffs) < 1e-4 else 'WARN - use in-run folds only'
print(f'  determinism vs round-3 constants: max abs diff {max(diffs):.5f} ({det_note})')
INC_FOLDS = aucs_inc

## Exp 35: TE on crossed columns

The Exp 31 scheme plus target encodings of Class x Type of Travel and Customer Type x Class,
built with the same split-first leak-free construction. The raw crossed strings are dropped -
only the encodings enter. Counts remain on the raw 21 columns. Stochastic config, seeds
42/43/44, OOF averaged per row.

In [ ]:
print('Exp 35: counts + all-21 TE + crossed-column TE + stochastic, seeds 42/43/44, averaged')
features_x = add_crosses(features)
TE_COLS_X = TE_COLS_ALL + CROSS_COLS
oof_seeds35 = {}
iters35 = None
for seed in [42, 43, 44]:
    o, a, it = oof_lgbm(features_x, target, f'E35-s{seed}', seed=seed,
                        count_features=True, target_encode=True,
                        te_cols=TE_COLS_X, te_drop=CROSS_COLS, **STOCH)
    oof_seeds35[seed] = o
    if seed == 42:
        iters35 = it
    print(f'  seed {seed} mean AUC: {np.mean(a):.5f}')
oof35 = np.mean([oof_seeds35[s] for s in [42, 43, 44]], axis=0)
aucs35 = fold_aucs(oof35, target)
print(f'Exp 35 (averaged) fold AUCs: {[round(a, 5) for a in aucs35]}')
print(f'Exp 35 mean OOF AUC: {np.mean(aucs35):.5f} +/- {np.std(aucs35):.5f} | pooled: {roc_auc_score(target, oof35):.6f}')

## Exp 36: config diversity on the TE base

The Exp 31 scheme with {num_leaves 63, 127} x 3 seeds - 6-model average. Exp 32 showed no
detected improvement from this diversity on the counts-only base; this retests it on the TE
base. Bounded to these two configs.

In [ ]:
print('Exp 36: counts + all-21 TE + stochastic, 3 seeds x {63, 127} leaves, 6-model average')
LEAF_CONFIGS = [63, 127]
oof_runs36, iters36 = {}, {}
for leaves in LEAF_CONFIGS:
    for seed in [42, 43, 44]:
        o, a, it = oof_lgbm(features, target, f'E36-l{leaves}-s{seed}', seed=seed,
                            count_features=True, target_encode=True,
                            num_leaves=leaves, **STOCH)
        oof_runs36[(leaves, seed)] = o
        iters36[(leaves, seed)] = it
        print(f'  leaves={leaves} seed {seed} mean AUC: {np.mean(a):.5f}')
oof36 = np.mean(list(oof_runs36.values()), axis=0)
aucs36 = fold_aucs(oof36, target)
print(f'Exp 36 (averaged) fold AUCs: {[round(a, 5) for a in aucs36]}')
print(f'Exp 36 mean OOF AUC: {np.mean(aucs36):.5f} +/- {np.std(aucs36):.5f} | pooled: {roc_auc_score(target, oof36):.6f}')

## Results, verdicts, and refit guard

Each candidate pairs against the in-run incumbent, the Exp 29 folds, and the Exp 31 folds (the
bar). Refit fires only on a WORKING SIGNAL better vs the Exp 31 bar; on a tie or loss, Exp 31 and
submission_round10.csv stand. Null results are bounded to the tested variant; a tie does not
establish equivalence.

In [ ]:
def paired_verdict(new_aucs, ref_aucs):
    deltas = [n - r for n, r in zip(new_aucs, ref_aucs)]
    mean_d, spread = float(np.mean(deltas)), float(np.std(deltas))
    if abs(mean_d) <= 2 * spread:
        label = 'NO SIGNAL (within 2x per-fold delta spread - no action)'
    else:
        label = ('WORKING SIGNAL: better' if mean_d > 0 else 'WORKING SIGNAL: worse') + ' (descriptive, not proof)'
    return mean_d, spread, label

results = {}
for tag, aucs, desc in [('35', aucs35, 'Exp 35: crossed-column TE'), ('36', aucs36, 'Exp 36: config diversity on TE base')]:
    print(desc)
    m_, s_, lab = paired_verdict(aucs, INC_FOLDS)
    print(f'  vs incumbent (in-run):         mean delta {m_:+.5f} (spread {s_:.5f}) - {lab}')
    m_, s_, lab = paired_verdict(aucs, E29_FOLDS_R9)
    print(f'  vs Exp 29 folds (round 9):     mean delta {m_:+.5f} (spread {s_:.5f}) - {lab}')
    m_bar, s_bar, lab_bar = paired_verdict(aucs, E31_FOLDS_R10)
    print(f'  vs Exp 31 folds (round 10 bar): mean delta {m_bar:+.5f} (spread {s_bar:.5f}) - {lab_bar}')
    results[tag] = (m_bar, lab_bar, float(np.mean(aucs)))

winners = [k for k, (m, lab, _) in results.items() if lab.startswith('WORKING SIGNAL: better')]
win = max(winners, key=lambda k: results[k][2]) if winners else None
print()
if win:
    print(f'REFIT GUARD: Exp {win} clears the Exp 31 bar under the paired band -> refit fires (exploratory, descriptive signal).')
else:
    print('REFIT GUARD: nothing cleared the Exp 31 bar under the paired band. Exp 31 remains the '
          'preferred candidate; submission_round10.csv is untouched.')

In [ ]:
if win is None:
    print('No submission file written this round.')
else:
    test = pd.read_csv(data_dir / 'test.csv')
    test_features = test.drop(columns=['id'])
    preds = []
    if win == '35':
        basis = int(round(np.mean(iters35)))
        feats_full, feats_test = add_crosses(features), add_crosses(test_features)
        tr_c, (te_c,) = add_counts(feats_full, [feats_test])
        te_full = te_crossfit(tr_c, target, TE_COLS_X)  # tr_c already carries the crossed columns
        te_test = te_apply(feats_full, target, feats_test, TE_COLS_X)
        X_full = pd.concat([tr_c.reset_index(drop=True), te_full.reset_index(drop=True)], axis=1)
        X_test = pd.concat([te_c.reset_index(drop=True), te_test.reset_index(drop=True)], axis=1)
        X_full = X_full.drop(columns=CROSS_COLS); X_test = X_test.drop(columns=CROSS_COLS)
        X_full, X_test = to_lgbm(X_full), to_lgbm(X_test)
        for seed in [42, 43, 44]:
            cfg = dict(learning_rate=0.05, num_leaves=63, n_estimators=basis,
                       n_jobs=-1, random_state=seed, verbose=-1)
            cfg.update(STOCH)
            model = lgb.LGBMClassifier(**cfg)
            model.fit(X_full, target)
            preds.append(model.predict_proba(X_test)[:, 1])
        scheme = 'Exp 35: counts + all-21 TE + crossed TE + stochastic, 3 seeds'
    else:  # win == '36'
        basis = int(round(np.mean([np.mean(iters36[(l, 42)]) for l in LEAF_CONFIGS])))
        tr_c, (te_c,) = add_counts(features, [test_features])
        te_full = te_crossfit(tr_c, target, TE_COLS_ALL)
        te_test = te_apply(features, target, test_features, TE_COLS_ALL)
        X_full = pd.concat([tr_c.reset_index(drop=True), te_full.reset_index(drop=True)], axis=1)
        X_test = pd.concat([te_c.reset_index(drop=True), te_test.reset_index(drop=True)], axis=1)
        X_full, X_test = to_lgbm(X_full), to_lgbm(X_test)
        for leaves in LEAF_CONFIGS:
            for seed in [42, 43, 44]:
                cfg = dict(learning_rate=0.05, num_leaves=leaves, n_estimators=basis,
                           n_jobs=-1, random_state=seed, verbose=-1)
                cfg.update(STOCH)
                model = lgb.LGBMClassifier(**cfg)
                model.fit(X_full, target)
                preds.append(model.predict_proba(X_test)[:, 1])
        scheme = 'Exp 36: counts + all-21 TE + stochastic, 3 seeds x {63,127} leaves'
    sub = pd.DataFrame({'id': test['id'], 'satisfaction': np.mean(preds, axis=0)})
    sub.to_csv('submission_round12.csv', index=False)
    print(f'submission_round12.csv written: {len(sub):,} rows | {scheme} | n_estimators basis {basis} '
          f'(in-run seed-42 fold-mean best_iteration). Local file only - NOT submitted; Andrea submits.')

In [ ]:
# OOF cache (established pattern): persist predictions + metadata for later rounds.
import hashlib, json as _json, time

cache_dir = competition_dir / 'oof_cache'
cache_dir.mkdir(exist_ok=True)
h = hashlib.sha256()
with open(competition_dir / 'data' / 'train.csv', 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 20), b''):
        h.update(chunk)
np.savez(cache_dir / 'round12_oof.npz',
         incumbent=oof_inc, exp35=oof35, exp36=oof36, target=target.to_numpy())
meta = {
    'round': 12,
    'train_sha256': h.hexdigest(),
    'created': time.strftime('%Y-%m-%d %H:%M:%S'),
    'configs': {
        'incumbent': {'note': 'Exp 8 config, in-run reference'},
        'exp35': {'scheme': 'Exp 31 + TE on Class x Type of Travel, Customer Type x Class (raw crosses dropped)'},
        'exp36': {'scheme': 'Exp 31 scheme, num_leaves {63,127} x seeds 42/43/44, 6-model average'},
        'stoch': STOCH,
    },
    'fold_aucs': {'incumbent': INC_FOLDS, 'exp35': aucs35, 'exp36': aucs36},
}
(cache_dir / 'round12_oof.json').write_text(_json.dumps(meta, indent=2))
print('cached oof_cache/round12_oof.npz + round12_oof.json')

## What happens next

- If Exp 35 beats the bar: interaction-level label information adds something on top of
  value-level TE in this test - follow-ups try one more crossed pair at a time, bounded.
- If Exp 36 beats the bar: config diversity earns its place on the TE base even though it did
  not on the counts-only base - the averaging scheme widens.
- If both are null: bounded to these tests (these two crosses, these two configs). The remaining
  levers get thinner: the pending external-data question (needs Andrea's explicit OK) becomes
  the strongest untried direction, and an honest plateau note for Andrea is worth drafting.